# YOLO Training — European License Plates
Fine-tunes a YOLOv8 model on the `images-eu` Roboflow dataset.

In [14]:
import os
import yaml
from pathlib import Path

# Resolve absolute paths so YOLO can find the data regardless of working directory
DATASET_ROOT = Path("images-eu").resolve()
YAML_FIXED = DATASET_ROOT / "data_fixed.yaml"

with open(DATASET_ROOT / "data.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["train"] = str(DATASET_ROOT / "train" / "images")
cfg["val"]   = str(DATASET_ROOT / "valid" / "images")
cfg["test"]  = str(DATASET_ROOT / "test"  / "images")

with open(YAML_FIXED, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("Fixed data.yaml written to:", YAML_FIXED)
print(yaml.dump(cfg, default_flow_style=False))
print ("Sizes of the datasets:")
print("Train:", len(list(Path(cfg["train"]).glob("*"))))
print("Validation:", len(list(Path(cfg["val"]).glob("*"))))
print("Test:", len(list(Path(cfg["test"]).glob("*"))))

Fixed data.yaml written to: C:\Users\krist\Documents\GitHub\OCR-Research\images-eu\data_fixed.yaml
names:
- '0'
nc: 1
roboflow:
  license: CC BY 4.0
  project: european-license-plates
  url: https://universe.roboflow.com/patrick-neicu/european-license-plates/dataset/1
  version: 1
  workspace: patrick-neicu
test: C:\Users\krist\Documents\GitHub\OCR-Research\images-eu\test\images
train: C:\Users\krist\Documents\GitHub\OCR-Research\images-eu\train\images
val: C:\Users\krist\Documents\GitHub\OCR-Research\images-eu\valid\images

Sizes of the datasets:
Train: 32
Validation: 28
Test: 13


## Train
Starts from `yolov8n.pt` (nano — fast to train). Adjust `model`, `epochs`, `imgsz`, and `batch` as needed.

In [15]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # pre-trained nano backbone; change to yolov8s.pt / yolov8m.pt for better accuracy

results = model.train(
    data=str(YAML_FIXED),
    epochs=10,
    imgsz=640,
    batch=16,
    name="eu_license_plates",
    project="train-result",
    exist_ok=True,
)

print("Training complete. Best weights saved to:", results.save_dir)

Ultralytics 8.4.41  Python-3.13.7 torch-2.7.1+cpu CPU (AMD Ryzen 7 PRO 7840U w/ Radeon 780M Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\krist\Documents\GitHub\OCR-Research\images-eu\data_fixed.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=eu_license_plates, nbs=64, nms=False, opset=

## Validate on the validation set

In [16]:
best_weights = Path("runs/detect/train-result/eu_license_plates/weights/best.pt")

trained_model = YOLO(str(best_weights))
metrics = trained_model.val(data=str(YAML_FIXED))

print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")

Ultralytics 8.4.41  Python-3.13.7 torch-2.7.1+cpu CPU (AMD Ryzen 7 PRO 7840U w/ Radeon 780M Graphics)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 427.6126.6 MB/s, size: 38.1 KB)
val: Scanning C:\Users\krist\Documents\GitHub\OCR-Research\images-eu\valid\labels.cache... 28 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 28/28 6.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.3it/s 1.5s2.9s
                   all         28         28    0.00333          1      0.579       0.49
Speed: 2.9ms preprocess, 41.7ms inference, 0.0ms loss, 3.1ms postprocess per image
Results saved to C:\Users\krist\Documents\GitHub\OCR-Research\runs\detect\val
mAP50:     0.5788
mAP50-95:  0.4902


## Quick inference on the test set
Runs the trained model on all test images and saves annotated results.

In [17]:
test_images = str(DATASET_ROOT / "test" / "images")

trained_model.predict(
    source=test_images,
    save=True,
    project="predict",
    name="eu_license_plates",
    exist_ok=True,
)

print("Predictions saved to runs/predict/eu_license_plates_test/")


image 1/13 C:\Users\krist\Documents\GitHub\OCR-Research\images-eu\test\images\d_license_plate_213_jpg.rf.16d04170b8c1fc9b8df121e5d0f1e3a9.jpg: 640x640 (no detections), 128.8ms
image 2/13 C:\Users\krist\Documents\GitHub\OCR-Research\images-eu\test\images\d_license_plate_217_jpg.rf.8e3b6839ee9df31899fceeecf3532e48.jpg: 640x640 (no detections), 53.9ms
image 3/13 C:\Users\krist\Documents\GitHub\OCR-Research\images-eu\test\images\d_license_plate_218_jpg.rf.114d77f6a5aad32adc92baf81a89ece6.jpg: 640x640 (no detections), 40.9ms
image 4/13 C:\Users\krist\Documents\GitHub\OCR-Research\images-eu\test\images\d_license_plate_243_jpg.rf.5a9a527d5e166a819515a93eee7ee089.jpg: 640x640 (no detections), 74.4ms
image 5/13 C:\Users\krist\Documents\GitHub\OCR-Research\images-eu\test\images\d_license_plate_250_jpg.rf.20972c4be87e4770dba72c2fe65695fa.jpg: 640x640 (no detections), 59.4ms
image 6/13 C:\Users\krist\Documents\GitHub\OCR-Research\images-eu\test\images\d_license_plate_280_jpg.rf.98248ef0a30c49bae1

## Update inference notebook to use the new model
Copy the best weights path so it can be used in `easy-ocr-prototype.ipynb`.

In [19]:
print("To use this model in easy-ocr-prototype.ipynb, load it with:")
print(f'  model = YOLO("{best_weights}")')

To use this model in easy-ocr-prototype.ipynb, load it with:
  model = YOLO("runs\detect\train-result\eu_license_plates\weights\best.pt")


## Run inference on the `images` folder
Uses the trained model to detect license plates in all images under `images/` and displays each result inline.

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

best_weights = Path("runs/detect/train-result/eu_license_plates/weights/best.pt")
trained_model = YOLO(str(best_weights))

images_dir = Path("images")
image_files = [p for p in images_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}]

results = trained_model.predict(
    source=str(images_dir),
    save=True,
    project="runs/detect",
    name="predict_images",
    exist_ok=True,
    conf=0.25,
)

print(f"Processed {len(results)} image(s). Annotated images saved to runs/detect/predict_images/")

# Display each annotated result inline
fig, axes = plt.subplots(1, len(results), figsize=(8 * len(results), 6))
if len(results) == 1:
    axes = [axes]

for ax, result in zip(axes, results):
    annotated = result.plot()  # BGR numpy array with boxes drawn
    ax.imshow(annotated[:, :, ::-1])  # convert BGR -> RGB
    ax.set_title(Path(result.path).name)
    ax.axis("off")
    detections = result.boxes
    if detections is not None and len(detections):
        print(f"{Path(result.path).name}: {len(detections)} detection(s)")
        for box in detections:
            cls_id = int(box.cls)
            conf = float(box.conf)
            label = trained_model.names[cls_id]
            print(f"  - {label}: {conf:.2%} confidence")
    else:
        print(f"{Path(result.path).name}: no detections")

plt.tight_layout()
plt.show()



image 1/2 c:\Users\krist\Documents\GitHub\OCR-Research\images\DSC_1939.JPG: 448x640 (no detections), 101.6ms
image 2/2 c:\Users\krist\Documents\GitHub\OCR-Research\images\DSC_1940.JPG: 448x640 (no detections), 46.5ms
Speed: 5.0ms preprocess, 74.0ms inference, 3.0ms postprocess per image at shape (1, 3, 448, 640)
Results saved to C:\Users\krist\Documents\GitHub\OCR-Research\runs\detect\runs\detect\predict_images
Processed 2 image(s). Annotated images saved to runs/detect/predict_images/
DSC_1939.JPG: no detections
DSC_1940.JPG: no detections


<Figure size 1600x600 with 2 Axes>

: 